In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'amazon-fine-food-reviews' dataset.
Path to dataset files: /kaggle/input/amazon-fine-food-reviews


In [2]:
import pandas as pd
import os

# Assuming the 'path' variable from the previous cell is available and contains the dataset directory.
# A common file in the 'amazon-fine-food-reviews' dataset is 'Reviews.csv'.
df = pd.read_csv(os.path.join(path, 'Reviews.csv'))

df.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,26
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,27
Text,0


In [3]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

# Convert Ratings to Sentiment Labels

In [4]:
def sentiment_label(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

df['Sentiment'] = df['Score'].apply(sentiment_label)

df[['Score', 'Sentiment']].head()

,Score,Sentiment
0,5,positive
1,1,negative
2,4,positive
3,2,negative
4,5,positive


In [5]:
df = df[['Text', 'Sentiment']]

df.head()

,Text,Sentiment
0,I have bought several of the Vitality canned d...,positive
1,Product arrived labeled as Jumbo Salted Peanut...,negative
2,This is a confection that has been around a fe...,positive
3,If you are looking for the secret ingredient i...,negative
4,Great taffy at a great price. There was a wid...,positive


# Check Class Distribution

In [6]:
df['Sentiment'].value_counts()

,count
Sentiment,
positive,443777
negative,82037
neutral,42640


# Reduce Dataset Size

In [7]:
df = df.sample(20000, random_state=42)

In [8]:
df = df.reset_index(drop=True)

# Basic Text Cleaning

In [9]:
!pip install nltk

In [10]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [12]:
df['Cleaned_Text'] = df['Text'].apply(clean_text)

df.head()

,Text,Sentiment,Cleaned_Text
0,Having tried a couple of other brands of glute...,positive,tried couple brands glutenfree sandwich cookie...
1,My cat loves these treats. If ever I can't fin...,positive,cat loves treats ever cant find house pop top ...
2,A little less than I expected. It tends to ha...,neutral,little less expected tends muddy taste expecte...
3,"First there was Frosted Mini-Wheats, in origin...",negative,first frosted miniwheats original size frosted...
4,and I want to congratulate the graphic artist ...,positive,want congratulate graphic artist putting entir...


# Encode Labels

In [13]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['Sentiment'])

df.head()

,Text,Sentiment,Cleaned_Text,label
0,Having tried a couple of other brands of glute...,positive,tried couple brands glutenfree sandwich cookie...,2
1,My cat loves these treats. If ever I can't fin...,positive,cat loves treats ever cant find house pop top ...,2
2,A little less than I expected. It tends to ha...,neutral,little less expected tends muddy taste expecte...,1
3,"First there was Frosted Mini-Wheats, in origin...",negative,first frosted miniwheats original size frosted...,0
4,and I want to congratulate the graphic artist ...,positive,want congratulate graphic artist putting entir...,2


# Train Test Split

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [15]:
train_data = pd.DataFrame({
    'text': X_train,
    'label': y_train
})

test_data = pd.DataFrame({
    'text': X_test,
    'label': y_test
})

train_data.to_csv("train_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)

In [17]:
from google.colab import files

files.download("train_data.csv")
files.download("test_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

MODEL TRAINING


In [20]:
import warnings
warnings.filterwarnings("ignore")

from transformers import logging
logging.set_verbosity_error()

# ── 1. BERT ──────────────────────────────────────────────────────────────────
from transformers import BertTokenizer, BertForSequenceClassification
import torch

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3,
    ignore_mismatched_sizes=True
)

def bert_tokenize(texts):
    return bert_tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')

bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)

def bert_train(dataloader, epochs=3):
    bert_model.train()
    for epoch in range(epochs):
        for batch in dataloader:
            bert_optimizer.zero_grad()
            outputs = bert_model(input_ids=batch['input_ids'],
                                 attention_mask=batch['attention_mask'],
                                 labels=batch['labels'])
            outputs.loss.backward()
            bert_optimizer.step()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]